# Topic 3 — Climate & Energy: Combined Data Cleaning

This notebook combines the existing EV charging, vehicle, and station cleaning workflow with the grid and weather hourly cleaning workflow.

**Part I:** EV charging datasets (original sections 1–14).

**Part II:** Grid and weather datasets, hourly merge, and daily averages (sections 15–22).

Run cells in order. Open the notebook from the project root or the Topic 3 `data` folder. Generated CSV files are written to the Topic 3 data folder.


# 1. Import Libraries and Load the Datasets

Pandas and NumPy are used to load, inspect, clean, and integrate the datasets.

Three datasets are loaded for this stage:

- **Charging Sessions:** Contains individual EV charging sessions, including the vehicle, station, start time, end time, charging type, and energy delivered.
- **Vehicles:** Contains information about the 10,000 EVs, including vehicle identifiers, battery capacity, registration month, and gasoline vehicle information.
- **Public Stations:** Contains information about public charging stations and their identifiers.

Identifiers are loaded as strings to preserve their original formatting. 

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Works when launched from the project root or the Topic 3 data folder.
project_data = Path("Topic3_Climate_Energy_The_Green_Tech_Carbon_Spike/data")
if not project_data.is_dir():
    project_data = Path(".")
if not (project_data / "charging_sessions.csv.gz").exists():
    raise FileNotFoundError("Open this notebook from the project root or Topic 3 data folder.")

# Load datasets while preserving IDs as text
charging = pd.read_csv(project_data / "charging_sessions.csv.gz", dtype={
    "session_id": "string",
    "vehicle_id": "string",
    "station_id": "string"
})

vehicles = pd.read_csv(project_data / "vehicles.csv", dtype={
    "vehicle_id": "string",
    "home_zip": "string"
})

stations = pd.read_csv(project_data / "stations.csv", dtype={
    "station_id": "string",
    "zip": "string"
})

print("Datasets loaded successfully!")

# 2. Record Original Row Counts

Before making any changes, the original number of rows in each dataset is recorded.

These counts provide a baseline for measuring how many records remain after cleaning.

The original and final row counts will be compared at the end of the notebook. Each cleaning operation is also recorded separately so that removed records and the reasons for their removal can be reported.

In [ ]:
original_counts = {
    "Charging Sessions": len(charging),
    "Vehicles": len(vehicles),
    "Public Stations": len(stations)
}

for name, count in original_counts.items():
    print(f"{name}: {count:,} rows")

# 3. Inspect the Original Datasets

The inspection reports:
- Dataset dimensions (rows and columns).
- Missing values in each column.
- Completely duplicated rows.
- Data types for each column.

This initial inspection helps identify problems that may affect later analysis, including missing energy measurements, duplicate records, incorrect data types, and incomplete identifiers.

The results are used to determine which cleaning steps are necessary.

In [ ]:
for name, df in {
    "Charging Sessions": charging,
    "Vehicles": vehicles,
    "Public Stations": stations
}.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("Shape:", df.shape)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nData types:")
    print(df.dtypes)

# 4. Create Cleaning and Data-Quality Logs

Two logs are created to document the cleaning process.

**Cleaning log:** Records operations that remove rows. For each operation, it records:
- Dataset name.
- Cleaning operation.
- Number of rows before cleaning.
- Number of rows after cleaning.
- Number of rows removed.
- Reason for the operation.

**Data-quality log:** Records invalid values or conversions that do not necessarily remove entire rows. For example, a value that cannot be converted into a number is changed to a missing value (`NaN`) and recorded.

Keeping these logs separate makes it possible to distinguish between records removed from the analysis and individual values that require further investigation.

In [ ]:
cleaning_log = []

def log_cleaning(dataset, step, before, after, reason):
    cleaning_log.append({
        "Dataset": dataset,
        "Cleaning Step": step,
        "Rows Before": before,
        "Rows After": after,
        "Rows Removed": before - after,
        "Reason": reason
    })

def show_cleaning_log():
    return pd.DataFrame(cleaning_log)

In [ ]:
# Track data-quality issues that do not necessarily remove rows
quality_log = []

def log_quality(dataset, column, affected, action, reason):
    quality_log.append({
        "Dataset": dataset,
        "Column": column,
        "Affected Values": affected,
        "Action": action,
        "Reason": reason
    })

def show_quality_log():
    return pd.DataFrame(quality_log)

# 5. Clean the Vehicle Dataset

The vehicle dataset is cleaned first because vehicle identifiers are needed to verify and connect charging sessions.

The following operations are performed:

1. **Standardize vehicle IDs:** Remove extra spaces and convert IDs to uppercase so that formatting differences do not prevent matches.
2. **Remove missing vehicle IDs:** Records without an ID cannot be reliably connected to charging sessions.
3. **Remove duplicate vehicle IDs:** Each vehicle should appear only once in the vehicle dataset. When duplicate IDs occur, the first record is retained.
4. **Convert numerical columns:** Battery capacity, rated energy consumption, and replaced gasoline vehicle MPG are converted to numeric values. Values that cannot be converted become `NaN` and are recorded in the data-quality log.
5. **Convert registration month:** Registration dates are converted into datetime values so vehicles can be analyzed according to when they entered the fleet. Invalid date values become `NaT` and are recorded.

The vehicle dataset is not reduced because of missing non-identifying vehicle information. Such values remain available for later investigation.

In [ ]:
vehicles_clean = vehicles.copy()

# Remove extra spaces and standardize IDs
vehicles_clean["vehicle_id"] = vehicles_clean["vehicle_id"].str.strip().str.upper()

# Remove records without vehicle IDs
before = len(vehicles_clean)
vehicles_clean = vehicles_clean.dropna(subset=["vehicle_id"])
log_cleaning(
    "Vehicles",
    "Remove missing vehicle IDs",
    before,
    len(vehicles_clean),
    "A vehicle must have an ID to connect it to charging sessions."
)

# Remove duplicate vehicle IDs
before = len(vehicles_clean)
vehicles_clean = vehicles_clean.drop_duplicates(
    subset=["vehicle_id"],
    keep="first"
)
log_cleaning(
    "Vehicles",
    "Remove duplicate vehicle IDs",
    before,
    len(vehicles_clean),
    "Each vehicle should appear only once."
)

numeric_columns = [
    "battery_kwh",
    "rated_kwh_per_100mi",
    "replaced_gas_car_mpg"
]

for column in numeric_columns:

    before_invalid = vehicles_clean[column].isna().sum()

    original_values = vehicles_clean[column].copy()

    vehicles_clean[column] = pd.to_numeric(
        vehicles_clean[column],
        errors="coerce"
    )

    newly_invalid = (
        original_values.notna() &
        vehicles_clean[column].isna()
    ).sum()

    log_quality(
        "Vehicles",
        column,
        newly_invalid,
        "Converted invalid numerical values to NaN",
        "The column must contain numeric values for calculations."
    )

original_registration = vehicles_clean["registration_month"].copy()

vehicles_clean["registration_month"] = pd.to_datetime(
    vehicles_clean["registration_month"],
    format="%Y-%m",
    errors="coerce"
)

invalid_registration = (
    original_registration.notna() &
    vehicles_clean["registration_month"].isna()
).sum()

log_quality(
    "Vehicles",
    "registration_month",
    invalid_registration,
    "Converted invalid dates to NaT",
    "Registration dates are needed to identify when vehicles entered the fleet."
)

print("Clean vehicle records:", len(vehicles_clean))
print("Unique vehicle IDs:", vehicles_clean["vehicle_id"].nunique())

# 6. Clean the Public Charging Station Dataset

The public station dataset is prepared so that public charging sessions can be matched to known stations.

The following operations are performed:

1. **Standardize station IDs:** Remove extra spaces and convert station IDs to uppercase.
2. **Remove missing station IDs:** A station without an identifier cannot be matched to a charging session.
3. **Remove duplicate station IDs:** Each public station should appear only once. When duplicate IDs occur, the first record is retained.
4. **Convert opening dates:** The station opening date is converted to a datetime value. Invalid dates become `NaT`.

Home charging stations are not expected to appear in this dataset because it contains public stations only. Home charging stations are checked separately using the vehicle identifier.

In [ ]:
stations_clean = stations.copy()

# Standardize station IDs
stations_clean["station_id"] = (
    stations_clean["station_id"].str.strip().str.upper()
)

# Remove missing station IDs
before = len(stations_clean)
stations_clean = stations_clean.dropna(subset=["station_id"])

log_cleaning(
    "Stations",
    "Remove missing station IDs",
    before,
    len(stations_clean),
    "A public station must have an ID to match charging sessions."
)

# Remove duplicate station IDs
before = len(stations_clean)
stations_clean = stations_clean.drop_duplicates(
    subset=["station_id"],
    keep="first"
)

log_cleaning(
    "Stations",
    "Remove duplicate station IDs",
    before,
    len(stations_clean),
    "Each public charging station should appear once."
)

# Convert station opening date
stations_clean["open_date"] = pd.to_datetime(
    stations_clean["open_date"],
    errors="coerce"
)

print("Clean public stations:", len(stations_clean))
print("Unique station IDs:", stations_clean["station_id"].nunique())

# 7. Clean the Charging Sessions Dataset

The charging sessions dataset contains the individual charging events used in the project's analysis. It is therefore important to preserve valid sessions while identifying incomplete or inconsistent records.

The following operations are performed:

1. **Standardize text fields:** Remove leading and trailing spaces from identifiers and descriptive columns.
2. **Standardize identifiers:** Convert session, vehicle, and station IDs to uppercase.
3. **Standardize charging location:** Convert the home/public field to lowercase so that the two categories can be consistently identified.
4. **Standardize charger information:** Convert port types to lowercase and connector types to uppercase.
5. **Convert energy measurements:** Convert `kwh` to a numeric value. Values that cannot be converted become `NaN` and are recorded in the data-quality log.
6. **Convert timestamps:** Convert session start and end times into datetime values. Invalid timestamps become `NaT` and are recorded.

The charging timestamps are recorded as local clock times without an explicit UTC offset. This creates a potential daylight-saving-time ambiguity during the fall clock change, when the 1:00 AM hour occurs twice.

For this reason, duration validation is postponed until the daylight-saving-time issue can be investigated. This avoids incorrectly removing potentially valid sessions.

In [ ]:
charging_clean = charging.copy()

# Standardize text columns
text_columns = [
    "session_id",
    "vehicle_id",
    "station_id",
    "port_type",
    "connector",
    "home_or_public"
]

for column in text_columns:
    charging_clean[column] = charging_clean[column].str.strip()

# Standardize identifiers
charging_clean["session_id"] = charging_clean["session_id"].str.upper()
charging_clean["vehicle_id"] = charging_clean["vehicle_id"].str.upper()
charging_clean["station_id"] = charging_clean["station_id"].str.upper()

# Standardize charging location
charging_clean["home_or_public"] = (
    charging_clean["home_or_public"].str.lower()
)

# Standardize charger types
charging_clean["port_type"] = (
    charging_clean["port_type"].str.lower()
)

charging_clean["connector"] = (
    charging_clean["connector"].str.upper()
)

original_kwh = charging_clean["kwh"].copy()

charging_clean["kwh"] = pd.to_numeric(
    charging_clean["kwh"],
    errors="coerce"
)

invalid_kwh = (
    original_kwh.notna() &
    charging_clean["kwh"].isna()
).sum()

log_quality(
    "Charging Sessions",
    "kwh",
    invalid_kwh,
    "Converted invalid numerical values to NaN",
    "Energy consumption must be numeric to calculate charging emissions."
)

# Convert start timestamps
original_start = charging_clean["start_time"].copy()

charging_clean["start_time"] = pd.to_datetime(
    charging_clean["start_time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

invalid_start = (
    original_start.notna() &
    charging_clean["start_time"].isna()
).sum()

log_quality(
    "Charging Sessions",
    "start_time",
    invalid_start,
    "Converted invalid timestamps to NaT",
    "Valid timestamps are required to determine when charging occurred."
)

# Convert end timestamps
original_end = charging_clean["end_time"].copy()

charging_clean["end_time"] = pd.to_datetime(
    charging_clean["end_time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

invalid_end = (
    original_end.notna() &
    charging_clean["end_time"].isna()
).sum()

log_quality(
    "Charging Sessions",
    "end_time",
    invalid_end,
    "Converted invalid timestamps to NaT",
    "Valid timestamps are required to calculate charging duration."
)

print("Charging sessions inspected:", len(charging_clean))

# 7.1 Preserve and Remove Sessions with Missing Essential Information

Before removing incomplete charging sessions, records with missing energy measurements are saved separately as `sessions_missing_energy.csv`.

These records are preserved for investigation because missing energy measurements cannot be used to calculate charging emissions. They are not automatically assigned an energy value of zero.

The notebook then reports the number of missing values in each required column.

The following fields are considered essential:
- Session ID.
- Vehicle ID.
- Station ID.
- Start time.
- End time.
- Energy delivered (`kwh`).
- Charging location (`home_or_public`).

Records missing any of these required values are removed from the main analysis dataset and the number removed is recorded.

This step ensures that the remaining sessions have the minimum information necessary for matching, time-based analysis, and energy calculations.

In [ ]:
required_columns = [
    "session_id",
    "vehicle_id",
    "station_id",
    "start_time",
    "end_time",
    "kwh",
    "home_or_public"
]

before = len(charging_clean)

missing_energy_sessions = charging_clean[
    charging_clean["kwh"].isna()
].copy()

missing_energy_sessions.to_csv(
    "sessions_missing_energy.csv",
    index=False
)

print(
    "Sessions saved for missing-energy investigation:",
    len(missing_energy_sessions)
)

print("MISSING REQUIRED VALUES BEFORE REMOVAL")
print("--------------------------------------")

for column in required_columns:
    print(f"{column}: {charging_clean[column].isna().sum():,}")
charging_clean = charging_clean.dropna(
    subset=required_columns
)

log_cleaning(
    "Charging Sessions",
    "Remove missing essential values",
    before,
    len(charging_clean),
    "Sessions need identifiers, timestamps, electricity consumption, and charging location."
)

# 7.2 Remove Duplicate Charging Sessions

Charging sessions are checked for duplicate session IDs.

Each session should represent one unique charging event. If the same session ID appears more than once, the notebook retains the first occurrence and removes subsequent occurrences.

This prevents the same charging event from being counted multiple times when calculating total energy consumption and emissions.

In [ ]:
before = len(charging_clean)

charging_clean = charging_clean.drop_duplicates(
    subset=["session_id"],
    keep="first"
)

log_cleaning(
    "Charging Sessions",
    "Remove duplicate session IDs",
    before,
    len(charging_clean),
    "Each charging session should appear only once."
)

# 7.3 Remove Invalid Energy and Charging Location Values

Charging sessions are checked for two additional problems:

1. **Invalid energy:** Sessions must have a positive energy value (`kwh > 0`).
2. **Invalid charging location:** The location must be identified as either `home` or `public`.

Session duration is intentionally not validated at this stage. Because the timestamps do not contain timezone offsets, a session crossing the repeated daylight-saving-time hour could appear to have a negative duration when interpreted as ordinary local timestamps. Duration validation will be addressed separately during timestamp and daylight-saving-time analysis.

In [ ]:
# Remove invalid energy and location values only.
# Duration validation will happen after DST ambiguity is investigated.

before = len(charging_clean)

charging_clean = charging_clean[
    (charging_clean["kwh"] > 0) &
    (charging_clean["home_or_public"].isin(["home", "public"]))
].copy()

log_cleaning(
    "Charging Sessions",
    "Remove invalid energy and location records",
    before,
    len(charging_clean),
    "Sessions must have positive energy and a recognized charging location. Duration validation is deferred until DST handling."
)

# 8. Match Charging Sessions to Vehicles

Each charging session must be associated with one of the vehicles in the vehicle dataset.

The vehicle IDs from the cleaned vehicle dataset are compared with the vehicle IDs in the cleaned charging sessions dataset.

Two groups are identified:

- **Matched sessions:** The vehicle ID exists in the vehicle dataset.
- **Unmatched sessions:** The vehicle ID does not exist in the vehicle dataset.

The number of matched and unmatched sessions is reported.

Sessions without a matching vehicle are removed from the main analysis dataset because they cannot be reliably associated with the EV fleet being studied. The number removed and the reason are recorded.

In [ ]:
# Identify sessions with no matching vehicle
vehicle_ids = set(vehicles_clean["vehicle_id"])

charging_clean["vehicle_match"] = (
    charging_clean["vehicle_id"].isin(vehicle_ids)
)

unmatched_vehicles = charging_clean[
    ~charging_clean["vehicle_match"]
].copy()

print("Sessions with matching vehicles:",
      charging_clean["vehicle_match"].sum())

print("Sessions without matching vehicles:",
      len(unmatched_vehicles))

In [ ]:
before = len(charging_clean)

charging_clean = charging_clean[
    charging_clean["vehicle_match"]
].copy()

charging_clean = charging_clean.drop(columns=["vehicle_match"])

log_cleaning(
    "Charging Sessions",
    "Remove sessions without matching vehicles",
    before,
    len(charging_clean),
    "Every analyzed charging session must belong to one of the 10,000 mandate vehicles."
)

# 9. Match Charging Sessions to Stations

Charging sessions are matched to stations using different rules for home and public charging.

**Public charging:**
- The session's station ID must exist in the cleaned public station dataset.

**Home charging:**
- Home station IDs follow the expected format `H-` followed by the vehicle ID.
- The session's station ID must match the expected home station ID for that vehicle.

Sessions without a valid station relationship are removed from the main analysis dataset. The number removed and the reason are recorded.


In [ ]:
public_station_ids = set(stations_clean["station_id"])

# Check public station matches
charging_clean["station_match"] = np.where(
    charging_clean["home_or_public"] == "public",
    charging_clean["station_id"].isin(public_station_ids),

    # Home station must match its vehicle ID
    charging_clean["station_id"] ==
    "H-" + charging_clean["vehicle_id"]
)

# Identify unmatched stations
unmatched_stations = charging_clean[
    ~charging_clean["station_match"]
].copy()

print("Sessions with matching stations:",
      charging_clean["station_match"].sum())

print("Sessions without matching stations:",
      len(unmatched_stations))

In [ ]:
before = len(charging_clean)

charging_clean = charging_clean[
    charging_clean["station_match"]
].copy()

charging_clean = charging_clean.drop(
    columns=["station_match"]
)

log_cleaning(
    "Charging Sessions",
    "Remove sessions without matching stations",
    before,
    len(charging_clean),
    "Public sessions must match stations.csv; home sessions must match their vehicle's H- station ID."
)

# 10. Check Charging Energy Against Vehicle Battery Capacity

The cleaned vehicle dataset is merged with the charging sessions dataset to add each vehicle's usable battery capacity.

Each session's delivered energy is then compared with the vehicle's battery capacity.

Sessions with delivered energy greater than the listed battery capacity are flagged as potential data-quality issues and saved to `battery_capacity_issues.csv`.

This data is not automatically removed. They are preserved for further investigation because an unusually large session may indicate an error, an unusual charging event, or another issue.

This check helps identify potentially unrealistic energy measurements before calculating total EV charging energy and emissions.

In [ ]:
# Add vehicle battery capacity to charging sessions
charging_with_battery = charging_clean.merge(
    vehicles[["vehicle_id", "battery_kwh"]],
    on="vehicle_id",
    how="left",
    validate="many_to_one"
)

# Identify sessions exceeding battery capacity
battery_issues = charging_with_battery[
    charging_with_battery["kwh"] > charging_with_battery["battery_kwh"]
].copy()

print("Sessions exceeding battery capacity:", len(battery_issues))
print("Sessions missing battery capacity:", charging_with_battery["battery_kwh"].isna().sum())

In [ ]:
battery_issues.to_csv(project_data / "battery_capacity_issues.csv",
    index=False
)

# 11. Separate Home and Public Charging Sessions

The cleaned charging sessions are divided into two datasets:

- **Home charging:** Sessions identified as home charging.
- **Public charging:** Sessions identified as public charging.

The number of sessions in each group is reported.

The notebook also checks that the combined number of home and public sessions equals the total number of cleaned sessions.

This separation is important for the research questions, especially the investigation of overnight home charging and its relationship to electricity demand and carbon emissions.

In [ ]:
# Separate home charging
home_charging = charging_clean[
    charging_clean["home_or_public"] == "home"
].copy()

# Separate public charging
public_charging = charging_clean[
    charging_clean["home_or_public"] == "public"
].copy()

print("Home charging sessions:", len(home_charging))
print("Public charging sessions:", len(public_charging))

print("Total separated sessions:",
      len(home_charging) + len(public_charging))

print("Total cleaned sessions:", len(charging_clean))

# 12. Integrate Vehicle and Station Information

Additional information is added to the home and public charging datasets using their corresponding identifiers.

For both datasets, vehicle information is merged using `vehicle_id`.

For public charging sessions, station information is also merged using `station_id`.

The merges use a many-to-one validation rule. This checks that multiple charging sessions can connect to one vehicle or station without creating unintended duplicate rows from the reference datasets.

The resulting datasets contain charging information along with relevant vehicle details. Public charging sessions also contain public station details.

This integration makes it possible to examine charging patterns by vehicle characteristics, charging location, and public station information.

In [ ]:
# Add vehicle information to home sessions
home_charging = home_charging.merge(
    vehicles_clean,
    on="vehicle_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_vehicle")
)

# Add vehicle and public station information
public_charging = public_charging.merge(
    vehicles_clean,
    on="vehicle_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_vehicle")
)

public_charging = public_charging.merge(
    stations_clean,
    on="station_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_station")
)

print("Home charging rows:", len(home_charging))
print("Public charging rows:", len(public_charging))

# 13. Save the Cleaned Datasets

The following files are created:

- `charging_sessions_clean.csv`: All cleaned and matched charging sessions.
- `home_charging_clean.csv`: Cleaned home charging sessions with vehicle information.
- `public_charging_clean.csv`: Cleaned public charging sessions with vehicle and station information.
- `vehicles_clean.csv`: Cleaned vehicle dataset.
- `stations_clean.csv`: Cleaned public station dataset.

Additional exception files are also saved:
- `sessions_missing_energy.csv`: Sessions with missing energy measurements.
- `battery_capacity_issues.csv`: Sessions with energy greater than the listed vehicle battery capacity.

These exception files preserve information that may be useful when investigating unusual patterns in the data.

In [ ]:
charging_clean.to_csv(project_data / "charging_sessions_clean.csv", index=False)

home_charging.to_csv(project_data / "home_charging_clean.csv", index=False)

public_charging.to_csv(project_data / "public_charging_clean.csv", index=False)

vehicles_clean.to_csv(project_data / "vehicles_clean.csv", index=False)

stations_clean.to_csv(project_data / "stations_clean.csv", index=False)

print("All cleaned datasets saved!")

# 14. Review Cleaning Results and Final Row Counts

The final section summarizes the results of the cleaning process.

The cleaning log reports the number of records removed during each operation and the reason for each removal.

The data-quality log reports invalid conversions and other affected values that did not necessarily result in row removal.

The notebook also compares the original and final row counts for each dataset and calculates the percentage of rows retained.

Finally, missing values in the cleaned datasets are reported.


In [ ]:
cleaning_report = show_cleaning_log()

display(cleaning_report)

Final before and after report

In [ ]:
# FINAL DATASET ROW COUNT COMPARISON

final_counts = {
    "Charging Sessions": len(charging_clean),
    "Vehicles": len(vehicles_clean),
    "Public Stations": len(stations_clean)
}

comparison = pd.DataFrame({
    "Dataset": original_counts.keys(),
    "Original Rows": original_counts.values(),
    "Final Rows": final_counts.values()
})

comparison["Rows Removed"] = (
    comparison["Original Rows"] - comparison["Final Rows"]
)

comparison["Percent Retained"] = (
    comparison["Final Rows"] /
    comparison["Original Rows"] * 100
).round(2)

display(comparison)

---

# Part II: Grid and Weather Hourly Data Cleaning

This part incorporates the grid and weather cleaning workflow, preserving the checks, corrections, time-zone alignment, hourly integration, and daily aggregation from `R3-DataCleaning-Grid-Hourly-Corrected.ipynb`. All cleaned outputs are saved alongside the Topic 3 source datasets.


# 15. Load and Inspect Grid and Weather Data

Load both source datasets, inspect the grid columns, and record the original row counts.


In [ ]:

import pandas as pd
from pathlib import Path

# Location of the original datasets
data_path = project_data

# Load the datasets
grid = pd.read_csv(data_path / "grid_hourly.csv")
weather = pd.read_csv(data_path / "weather_hourly.csv")

# Record original row counts
print("Original grid rows:", len(grid))
print("Original weather rows:", len(weather))

# Preview both datasets
display(grid.head())
display(weather.head())


In [ ]:
print("Original grid rows:", len(grid))
print("\nColumn names:")
print(grid.columns.tolist())

display(grid.head())

# 16. Inspect Grid Data Quality

Check missing values, duplicates, and negative demand values before applying corrections.


In [ ]:

# Check for missing values
print("MISSING VALUES:")
print(grid.isnull().sum())

# Check for duplicate rows
print("\nDUPLICATE ROWS:")
print(grid.duplicated().sum())

# Check for negative electricity demand
print("\nNEGATIVE DEMAND VALUES:")
print((grid["demand_mwh"] < 0).sum())

# Check the different units
print("\nUNIT LABELS:")
print(grid["unit"].value_counts(dropna=False))

# Total rows before cleaning
print("\nTOTAL ROWS:")
print(len(grid))


In [ ]:

# Find all rows with negative electricity demand
negative_demand = grid[grid["demand_mwh"] < 0]

print("Number of negative demand rows:", len(negative_demand))

# Display the affected records
display(negative_demand[
    ["utc_time_at_end_of_hour",
     "demand_mwh",
     "net_generation",
     "unit"]
])


In [ ]:

# Compare negative demand with forecasts
negative_demand = grid[grid["demand_mwh"] < 0].copy()

negative_demand["absolute_demand"] = (
    negative_demand["demand_mwh"].abs()
)

display(negative_demand[
    ["utc_time_at_end_of_hour",
     "demand_forecast_mwh",
     "demand_mwh",
     "absolute_demand",
     "net_generation"]
])


# 17. Correct Grid Demand and Generation Values

Apply the original grid corrections, including wind gaps and generation-unit standardization; verify the result.


In [ ]:

# Create a cleaned copy of the grid data
grid_clean = grid.copy()

# Count negative values before correction
negative_count = (grid_clean["demand_mwh"] < 0).sum()

# Correct negative demand values
grid_clean["demand_mwh"] = grid_clean["demand_mwh"].abs()

# Verify the correction
print("Original rows:", len(grid))
print("Negative values corrected:", negative_count)
print("Rows removed:", len(grid) - len(grid_clean))
print("Remaining rows:", len(grid_clean))
print("Negative values remaining:",
      (grid_clean["demand_mwh"] < 0).sum())


In [ ]:

# Find rows with missing wind generation
missing_wind = grid_clean[grid_clean["wind"].isna()].copy()

print("Missing wind values:", len(missing_wind))

# Display the first 10 affected records
display(missing_wind[
    ["utc_time_at_end_of_hour",
     "net_generation",
     "coal",
     "natural_gas",
     "nuclear",
     "wind",
     "solar",
     "hydro",
     "other"]
].head(10))


In [ ]:

# Generation sources
sources = [
    "coal", "natural_gas", "nuclear",
    "wind", "solar", "hydro", "other"
]

# Check rows where wind is already known
known_wind = grid_clean[grid_clean["wind"].notna()].copy()

# Calculate the sum of generation sources
known_wind["source_total"] = known_wind[sources].sum(axis=1)

# Compare against reported net generation
known_wind["difference"] = (
    known_wind["net_generation"] - known_wind["source_total"]
)

print("Rows checked:", len(known_wind))
print("Exact matches:", (known_wind["difference"] == 0).sum())
print("Within 2 MWh:", (known_wind["difference"].abs() <= 2).sum())
print("Largest difference:", known_wind["difference"].abs().max())

display(known_wind[
    ["net_generation", "source_total", "difference"]
].head(10))


In [ ]:

# List generation sources other than wind
other_sources = [
    "coal", "natural_gas", "nuclear",
    "solar", "hydro", "other"
]

# Find rows with missing wind values
missing_wind = grid_clean[grid_clean["wind"].isna()].copy()

# Calculate missing wind generation
missing_wind["calculated_wind"] = (
    missing_wind["net_generation"]
    - missing_wind[other_sources].sum(axis=1)
)

# Check calculated values
print("Missing wind values:", len(missing_wind))
print("Negative calculated values:",
      (missing_wind["calculated_wind"] < 0).sum())
print("Minimum calculated wind:",
      missing_wind["calculated_wind"].min())
print("Maximum calculated wind:",
      missing_wind["calculated_wind"].max())

display(missing_wind[
    ["utc_time_at_end_of_hour", "calculated_wind"]
].head(10))


In [ ]:

# Count missing values before filling
missing_before = grid_clean["wind"].isna().sum()

# Fill missing wind values using calculated values
grid_clean.loc[missing_wind.index, "wind"] = (
    missing_wind["calculated_wind"]
)

# Verify the results
print("Wind values filled:", missing_before)
print("Missing wind values remaining:",
      grid_clean["wind"].isna().sum())
print("Rows remaining:", len(grid_clean))


In [ ]:
# Inspect original generation values by recorded unit (before conversion)
print("ORIGINAL UNIT LABEL COUNTS:")
print(grid["unit"].value_counts(dropna=False).to_string())
print("\nNET GENERATION BY ORIGINAL UNIT:")
display(grid.groupby("unit", dropna=False)["net_generation"].agg(["count", "min", "median", "max"]))


In [ ]:
# Convert generation and interchange quantities into MWh.
# Data dictionary: unit applies to net_generation, total_interchange, and fuel columns;
# demand_mwh and demand_forecast_mwh are already in MWh.
generation_columns = [
    "net_generation", "total_interchange", "coal", "natural_gas",
    "nuclear", "wind", "solar", "hydro", "other"
]
gwh_mask = grid_clean["unit"].eq("GWh")
missing_units = grid_clean["unit"].isna().sum()
gwh_rows = gwh_mask.sum()

# GWh values are 1/1000 of comparable MWh values in the source data.
grid_clean.loc[gwh_mask, generation_columns] *= 1000
# The blank-unit rows are on the MWh scale; standardize their metadata too.
grid_clean["unit"] = "MWh"

print("GWh rows converted to MWh:", gwh_rows)
print("Blank unit labels standardized to MWh:", missing_units)
print("Remaining missing unit labels:", grid_clean["unit"].isna().sum())
print("Units after conversion:")
print(grid_clean["unit"].value_counts().to_string())
print("\nNet generation after conversion:")
print(grid_clean["net_generation"].agg(["min", "median", "max"]))


In [ ]:

# Final grid validation
print("Total rows:", len(grid_clean))

print("\nMissing values:")
print(grid_clean.isna().sum().to_string())

print("\nNegative demand values:",
      (grid_clean["demand_mwh"] < 0).sum())

print("Duplicate timestamps:",
      grid_clean["utc_time_at_end_of_hour"].duplicated().sum())

print("\nDate range:")
print("First:", grid_clean["utc_time_at_end_of_hour"].min())
print("Last:", grid_clean["utc_time_at_end_of_hour"].max())


# 18. Inspect and Clean Weather Data

Inspect duplicate timestamps, fill missing measurements, and validate the cleaned weather records.


In [ ]:

# Check weather dataset for problems
print("Original weather rows:", len(weather))

print("\nMissing values:")
print(weather.isna().sum().to_string())

print("\nDuplicate rows:", weather.duplicated().sum())

print("Duplicate timestamps:",
      weather["timestamp"].duplicated().sum())

print("\nDate range:")
print("First:", weather["timestamp"].min())
print("Last:", weather["timestamp"].max())


In [ ]:

# Find weather records with repeated timestamps
duplicate_times = weather[
    weather["timestamp"].duplicated(keep=False)
].copy()

print("Records with repeated timestamps:", len(duplicate_times))

# Display the affected records
display(duplicate_times.sort_values("timestamp"))


In [ ]:

# Create a cleaned copy of the weather dataset
weather_clean = weather.copy()

# Check missing temperature and heating degree hours
missing_temp = weather_clean["temp_f"].isna()
missing_heating = weather_clean["heating_degree_hours"].isna()

print("Missing temperatures:", missing_temp.sum())
print("Missing heating degree hours:", missing_heating.sum())

print("Rows missing both:",
      (missing_temp & missing_heating).sum())

print("Rows missing only temperature:",
      (missing_temp & ~missing_heating).sum())

print("Rows missing only heating degree hours:",
      (~missing_temp & missing_heating).sum())

# Display first 10 affected records
display(weather_clean[
    missing_temp | missing_heating
].head(10))


In [ ]:

# Identify consecutive groups of missing temperatures
missing = weather_clean["temp_f"].isna()

group_ids = (missing != missing.shift()).cumsum()

missing_groups = (
    weather_clean[missing]
    .groupby(group_ids[missing])
    .size()
)

print("Total missing temperatures:", missing.sum())
print("Number of missing groups:", len(missing_groups))
print("Longest consecutive missing group:", missing_groups.max())

print("\nGroup sizes:")
print(missing_groups.value_counts().sort_index())


In [ ]:

# Count missing temperatures before cleaning
missing_before = weather_clean["temp_f"].isna().sum()

# Fill missing temperatures using linear interpolation
weather_clean["temp_f"] = (
    weather_clean["temp_f"].interpolate(
        method="linear",
        limit=1,
        limit_area="inside"
    )
)

# Verify the results
print("Temperatures filled:", missing_before)
print("Missing temperatures remaining:",
      weather_clean["temp_f"].isna().sum())
print("Rows remaining:", len(weather_clean))


In [ ]:

# Check the heating degree hour formula
known = weather_clean["heating_degree_hours"].notna()

calculated = (
    65 - weather_clean.loc[known, "temp_f"]
).clip(lower=0)

difference = (
    weather_clean.loc[known, "heating_degree_hours"]
    - calculated
).abs()

print("Existing records checked:", known.sum())
print("Records matching formula:", (difference < 0.01).sum())
print("Largest difference:", difference.max())
print("Missing heating degree hours:",
      weather_clean["heating_degree_hours"].isna().sum())


In [ ]:

# Identify missing heating degree hours
missing_heating = weather_clean["heating_degree_hours"].isna()

# Calculate missing values from temperature
weather_clean.loc[missing_heating, "heating_degree_hours"] = (
    65 - weather_clean.loc[missing_heating, "temp_f"]
).clip(lower=0)

# Verify the results
print("Heating degree hours filled:", missing_heating.sum())
print("Missing values remaining:",
      weather_clean["heating_degree_hours"].isna().sum())
print("Rows remaining:", len(weather_clean))


In [ ]:

# Final weather validation
print("Total rows:", len(weather_clean))

print("\nMissing values:")
print(weather_clean.isna().sum().to_string())

print("\nTemperature range:")
print("Minimum:", weather_clean["temp_f"].min())
print("Maximum:", weather_clean["temp_f"].max())

print("\nHumidity range:")
print("Minimum:", weather_clean["relative_humidity"].min())
print("Maximum:", weather_clean["relative_humidity"].max())

print("\nNegative wind speeds:",
      (weather_clean["wind_speed_mph"] < 0).sum())

print("Duplicate timestamps:",
      weather_clean["timestamp"].duplicated().sum())


# 19. Align Grid and Weather Timestamps

Check daylight-saving transitions and convert local weather timestamps to UTC before matching hour-ending grid timestamps.


In [ ]:

# Display grid timestamps around daylight saving time
grid_times = pd.to_datetime(
    grid_clean["utc_time_at_end_of_hour"]
)

weather_times = pd.to_datetime(
    weather_clean["timestamp"]
)

print("GRID - November 6, 2022:")
display(grid_clean.loc[
    grid_times.between("2022-11-06 05:00", "2022-11-06 10:00"),
    ["utc_time_at_end_of_hour", "demand_mwh"]
])

print("\nWEATHER - November 6, 2022:")
display(weather_clean.loc[
    weather_times.between("2022-11-06 00:00", "2022-11-06 04:00"),
    ["timestamp", "temp_f"]
])


In [ ]:

# Convert weather timestamps to UTC
weather_clean["local_time"] = pd.to_datetime(
    weather_clean["timestamp"]
)

weather_clean["weather_utc"] = (
    weather_clean["local_time"]
    .dt.tz_localize(
        "America/Chicago",
        ambiguous="infer",
        nonexistent="NaT"
    )
    .dt.tz_convert("UTC")
)

# Check the converted timestamps
print("Missing UTC timestamps:",
      weather_clean["weather_utc"].isna().sum())

print("Duplicate UTC timestamps:",
      weather_clean["weather_utc"].duplicated().sum())

print("\nConverted daylight saving timestamps:")
display(weather_clean.loc[
    weather_clean["timestamp"].isin([
        "2022-11-06 01:00",
        "2023-11-05 01:00"
    ]),
    ["timestamp", "weather_utc"]
])


In [ ]:

# Convert grid timestamps to UTC
grid_clean["grid_utc"] = pd.to_datetime(
    grid_clean["utc_time_at_end_of_hour"],
    utc=True
)

# Check alignment with weather timestamps
grid_hours = set(grid_clean["grid_utc"])
weather_hours = set(weather_clean["weather_utc"])

# Option 1: Match timestamps directly
direct_matches = len(grid_hours & weather_hours)

# Option 2: Shift weather timestamps forward one hour
shifted_weather_hours = {
    t + pd.Timedelta(hours=1)
    for t in weather_hours
}

shifted_matches = len(grid_hours & shifted_weather_hours)

print("Direct timestamp matches:", direct_matches)
print("Matches with weather shifted +1 hour:", shifted_matches)
print("Total grid records:", len(grid_clean))
print("Total weather records:", len(weather_clean))


# 20. Merge Hourly Grid and Weather Data

Merge the cleaned datasets by aligned UTC timestamps and validate the one-to-one match.


In [ ]:

# Align weather timestamps with grid hour-ending timestamps
weather_clean["matching_utc"] = (
    weather_clean["weather_utc"] + pd.Timedelta(hours=1)
)

# Merge grid and weather datasets
combined_hourly = pd.merge(
    grid_clean,
    weather_clean,
    left_on="grid_utc",
    right_on="matching_utc",
    how="inner",
    validate="one_to_one"
)

# Verify the merge
print("Original grid rows:", len(grid_clean))
print("Original weather rows:", len(weather_clean))
print("Combined hourly rows:", len(combined_hourly))
print("Missing values:", combined_hourly.isna().sum().sum())
print("Duplicate UTC timestamps:",
      combined_hourly["grid_utc"].duplicated().sum())

# Preview combined data
display(combined_hourly[
    ["grid_utc", "demand_mwh", "temp_f",
     "wind", "coal", "natural_gas"]
].head())


# 21. Calculate Daily Averages

Aggregate the merged hourly observations by local calendar date and validate the hourly totals.


In [ ]:

# Use the weather observation's local calendar date
combined_hourly["date"] = (
    pd.to_datetime(combined_hourly["timestamp"]).dt.date
)

# Columns to average for each day
average_columns = [
    "demand_mwh",
    "demand_forecast_mwh",
    "net_generation",
    "coal",
    "natural_gas",
    "nuclear",
    "wind",
    "solar",
    "hydro",
    "other",
    "temp_f",
    "heating_degree_hours",
    "wind_speed_mph",
    "relative_humidity"
]

# Calculate daily averages and count hourly records
daily_averages = (
    combined_hourly.groupby("date")
    .agg(
        **{col: (col, "mean") for col in average_columns},
        hours_recorded=("grid_utc", "count")
    )
    .reset_index()
)

# Check results
print("Original hourly records:", len(combined_hourly))
print("Daily records:", len(daily_averages))

print("\nHours per day:")
print(daily_averages["hours_recorded"].value_counts().sort_index())

display(daily_averages.head())


In [ ]:

# Verify daily aggregation
print("Daily records:", len(daily_averages))

print("Total hours accounted for:",
      daily_averages["hours_recorded"].sum())

print("Original hourly records:",
      len(combined_hourly))

print("Missing daily values:",
      daily_averages.isna().sum().sum())

print("Duplicate dates:",
      daily_averages["date"].duplicated().sum())

print("\nDate range:")
print("First:", daily_averages["date"].min())
print("Last:", daily_averages["date"].max())

# Check that the daily averages preserve the overall demand
weighted_demand = (
    (daily_averages["demand_mwh"] *
     daily_averages["hours_recorded"]).sum()
    / daily_averages["hours_recorded"].sum()
)

print("\nOriginal hourly demand average:",
      combined_hourly["demand_mwh"].mean())

print("Weighted daily demand average:",
      weighted_demand)


# 22. Save Cleaned Hourly and Daily Datasets

Write the cleaned hourly and daily datasets to the Topic 3 data directory.


In [ ]:

# Save cleaned datasets to the project data folder
output_folder = data_path

# Save hourly dataset
combined_hourly.to_csv(
    output_folder / "combined_hourly_clean.csv",
    index=False
)

# Save daily averages
daily_averages.to_csv(
    output_folder / "combined_daily_averages.csv",
    index=False
)

# Confirm saved files
print("Saved combined_hourly_clean.csv")
print("Saved combined_daily_averages.csv")
print("Hourly rows:", len(combined_hourly))
print("Daily rows:", len(daily_averages))
